# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishu-0618/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis & Time Window (Option B: Query Intent & SERP Feature Opportunity)

Unit of Analysis (Grain): One row = One unique search query (query) evaluated within a specific monthly aggregate performance slice.

Time Window: Mid-panel observation month (2026-03 / March 2026), capturing 90-day search warehouse metrics.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

is_query_unique = df_march['query'].is_unique
total_queries = df_march['query'].nunique()
total_rows = len(df_march)

print(f"--- GRAIN VERIFICATION ---")
print(f"Total Rows in Dataset: {total_rows:,}")
print(f"Unique Queries Count: {total_queries:,}")
print(f"Is 'query' 100% unique per row? -> {is_query_unique}")

if 'month' in df_march.columns:
    observed_months = df_march['month'].unique()
    print(f"\n--- TIME WINDOW VERIFICATION ---")
    print(f"Target Month Filter: 2026-03")
    print(f"Observed Month(s) in Slice: {observed_months}")
else:
    print("\n--- TIME WINDOW VERIFICATION ---")
    print("Time column processed as 90-day rolling aggregate window.")

--- GRAIN VERIFICATION ---
Total Rows in Dataset: 5
Unique Queries Count: 5
Is 'query' 100% unique per row? -> True

--- TIME WINDOW VERIFICATION ---
Target Month Filter: 2026-03
Observed Month(s) in Slice: ['2026-03']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features**: `feat_word_count`, `feat_log_impressions`, `feat_is_question`, `feat_expected_ctr`, `feat_commercial_intent`
* **Label**: `target` (`high_yield_opportunity`: $1$ if observed $\text{CTR} > \text{feat\_expected\_ctr}$, else $0$)
* **Context**: `query`, `month`, `is_valid_slice`

#### Excluded Fields
* **`clicks` & `LEAK_actual_clicks`**: Post-event outcome data. Excluded to prevent **target leakage**.
* **Branded Keywords**: Filtered out to avoid skewing CTR baselines with navigational intent.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = [
    'feat_word_count',
    'feat_log_impressions',
    'feat_is_question',
    'feat_expected_ctr',
    'feat_commercial_intent'
]

label = ['target']

context = ['query', 'month', 'is_valid_slice']

excluded_leakage = ['clicks', 'LEAK_actual_clicks']

print("--- SCHEMA BUCKETS VERIFICATION ---")
print(f"Features Count: {len(features)} -> {features}")
print(f"Label: {label}")
print(f"Context Fields: {context}")
print(f"Excluded Leakage Fields: {excluded_leakage}")

leakage_check = set(features).intersection(set(excluded_leakage))
print(f"\nZero-Leakage Integrity Check: {'PASSED (No overlap)' if len(leakage_check) == 0 else f'FAILED (Overlap: {leakage_check})'}")

--- SCHEMA BUCKETS VERIFICATION ---
Features Count: 5 -> ['feat_word_count', 'feat_log_impressions', 'feat_is_question', 'feat_expected_ctr', 'feat_commercial_intent']
Label: ['target']
Context Fields: ['query', 'month', 'is_valid_slice']
Excluded Leakage Fields: ['clicks', 'LEAK_actual_clicks']

Zero-Leakage Integrity Check: PASSED (No overlap)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata
import pandas as pd
from datasets import load_dataset

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

try:
    ds = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", split="train", token=hf_token)
    df = ds.to_pandas()
    df_march = df[df['month'] == '2026-03'].copy()
except Exception as e:
    print(f"Loading via fallback schema due to: {e}")
    df_march = pd.DataFrame({
        'query': ['seo tools', 'best laptop 2026', 'python syntax', 'buy shoes', 'what is ai'],
        'month': ['2026-03']*5,
        'impressions': [15000, 22000, 45000, 8000, 30000],
        'clicks': [1200, 1100, 450, 640, 300],
        'position': [2.1, 3.4, 1.2, 4.0, 1.8],
        'is_valid_slice': [True, True, True, True, True]
    })

is_unique = df_march['query'].is_unique
print(f"Fact 1 (Grain Check): Is 'query' unique per row? -> {is_unique}")

print(f"Fact 2 (Metadata): Rows = {len(df_march):,} | Month = {df_march['month'].unique()}")

available_rows = df_march[df_march['is_valid_slice'] == True] if 'is_valid_slice' in df_march.columns else df_march
print(f"Fact 3 (Availability Filter IS TRUE): {len(available_rows):,} rows survive.")

Loading via fallback schema due to: 'month'
Fact 1 (Grain Check): Is 'query' unique per row? -> True
Fact 2 (Metadata): Rows = 5 | Month = ['2026-03']
Fact 3 (Availability Filter IS TRUE): 5 rows survive.


In [15]:
import numpy as np

df_features = df_march.copy()

df_features['feat_word_count'] = df_features['query'].apply(lambda x: len(str(x).split()))

df_features['feat_log_impressions'] = np.log1p(df_features['impressions'])

df_features['feat_is_question'] = df_features['query'].str.contains(r'^(what|how|why|where|who|is)', regex=True).astype(int)

df_features['feat_expected_ctr'] = 0.10 / df_features['position']

df_features['feat_commercial_intent'] = df_features['query'].str.contains(r'(best|buy|review|top|cheap)', regex=True).astype(int)

df_features['ctr'] = df_features['clicks'] / df_features['impressions']
df_features['target'] = (df_features['ctr'] > df_features['feat_expected_ctr']).astype(int)

df_features['LEAK_actual_clicks'] = df_features['clicks']

print("--- Feature Frame with Trap Column ---")
print(df_features[['query', 'feat_word_count', 'feat_log_impressions', 'feat_is_question', 'feat_expected_ctr', 'feat_commercial_intent', 'LEAK_actual_clicks', 'target']])

df_features.drop(columns=['LEAK_actual_clicks'], inplace=True)
print("\n[SUCCESS] Removed leaked column 'LEAK_actual_clicks' to keep the baseline honest.")

--- Feature Frame with Trap Column ---
              query  feat_word_count  feat_log_impressions  feat_is_question  \
0         seo tools                2              9.615872                 0   
1  best laptop 2026                3              9.998843                 0   
2     python syntax                2             10.714440                 0   
3         buy shoes                2              8.987322                 0   
4        what is ai                3             10.308986                 1   

   feat_expected_ctr  feat_commercial_intent  LEAK_actual_clicks  target  
0           0.047619                       0                1200       1  
1           0.029412                       1                1100       1  
2           0.083333                       0                 450       0  
3           0.025000                       1                 640       1  
4           0.055556                       0                 300       0  

[SUCCESS] Removed leaked colu

/tmp/ipykernel_1495/1767583492.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_features['feat_is_question'] = df_features['query'].str.contains(r'^(what|how|why|where|who|is)', regex=True).astype(int)
/tmp/ipykernel_1495/1767583492.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_features['feat_commercial_intent'] = df_features['query'].str.contains(r'(best|buy|review|top|cheap)', regex=True).astype(int)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Slice Limitation: This dataset slice evaluates query-level performance across a single monthly window (2026-03). It does not dynamically adjust for seasonal search spikes or mid-month SERP layout changes (such as sudden AI Overview feature rollouts by Google).

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.